# Kaggle Production Pipeline: TD3 + DDPG

Notebook flow:
1. Setup
2. Config
3. Batch selection
4. Robust training loop
5. Conditional plotting
6. Export

This notebook is fault-tolerant and resume-safe:
- completed experiments are skipped
- failed experiments are isolated
- plotting runs only after all 24 experiments complete

In [ ]:
import os
import sys
import glob
import json
import shutil
import subprocess
from pathlib import Path

BASE_DIR = "/kaggle/working"
PROJECT_DIR = Path.cwd()
os.chdir(PROJECT_DIR)

print("Project directory:", PROJECT_DIR)
print("BASE_DIR:", BASE_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

import torch
print("GPU Available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

for folder in ["logs", "models", "results"]:
    Path(BASE_DIR, folder).mkdir(parents=True, exist_ok=True)
    Path(PROJECT_DIR, folder).mkdir(parents=True, exist_ok=True)

print(Path(BASE_DIR, "logs"))
print(Path(BASE_DIR, "models"))
print(Path(BASE_DIR, "results"))

In [ ]:
from config import EXPERIMENTS, EXPERIMENT_REWARD_MODES, EXPERIMENT_SENSOR_NOISE_LEVELS

MAX_EPISODES = 5000
HEADLESS = True
ALGORITHMS = ["td3", "ddpg"]

BATCHES = [
    ("td3", 0, 6),
    ("td3", 6, 6),
    ("ddpg", 0, 6),
    ("ddpg", 6, 6),
]

assert len(EXPERIMENTS) == 12
assert len(BATCHES) == 4

def build_experiment_tags():
    tags = []
    for reward_mode in EXPERIMENT_REWARD_MODES:
        for noise in EXPERIMENT_SENSOR_NOISE_LEVELS:
            r_idx = EXPERIMENT_REWARD_MODES.index(reward_mode) + 1
            n_idx = EXPERIMENT_SENSOR_NOISE_LEVELS.index(noise) + 1
            tags.append(f"R{r_idx}_N{n_idx}")
    return tags

EXPERIMENT_TAGS = build_experiment_tags()

def _read_last_log_entry(log_file: Path):
    if not log_file.exists():
        return None
    last = None
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                last = json.loads(line)
            except json.JSONDecodeError:
                continue
    return last

def _last_logged_episode(algo, exp_id):
    log_file = Path(BASE_DIR, "logs", algo, exp_id, "training_log.jsonl")
    if not log_file.exists():
        return 0
    max_ep = 0
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                payload = json.loads(line)
            except json.JSONDecodeError:
                continue
            ep = int(payload.get("episode", 0) or 0)
            if ep > max_ep:
                max_ep = ep
    return max_ep

def is_experiment_complete(algo, exp_id, max_episodes=5000):
    model_dir = Path(BASE_DIR, "models", algo, exp_id)
    best_models = list(model_dir.glob("*_best.pth")) if model_dir.exists() else []
    if best_models:
        return True
    return _last_logged_episode(algo, exp_id) >= int(max_episodes)

def get_latest_checkpoint(algo, exp_id):
    model_dir = Path(BASE_DIR, "models", algo, exp_id)
    if not model_dir.exists():
        return None
    ckpts = []
    for p in model_dir.glob("*_ep*.pth"):
        stem = p.stem
        try:
            ep = int(stem.split("_ep")[-1])
        except ValueError:
            continue
        ckpts.append((ep, p))
    if not ckpts:
        return None
    ckpts.sort(key=lambda x: x[0], reverse=True)
    return ckpts[0][1]

def prune_experiment_models(algo, exp_id):
    model_dir = Path(BASE_DIR, "models", algo, exp_id)
    if not model_dir.exists():
        return

    best_candidates = sorted(model_dir.glob("*_best.pth"))
    if not best_candidates:
        best_candidates = sorted(model_dir.glob("*_best_avg100.pth"))
    best_keep = best_candidates[0] if best_candidates else None

    latest_ckpt = get_latest_checkpoint(algo, exp_id)

    for path in model_dir.glob("*.pth"):
        if best_keep and path == best_keep:
            continue
        if latest_ckpt and path == latest_ckpt:
            continue
        if path.name.endswith("_best_avg100.pth") and best_keep and path != best_keep:
            path.unlink(missing_ok=True)
            continue
        if "_ep" in path.stem:
            path.unlink(missing_ok=True)

def last_progress_line(algo, exp_id):
    log_file = Path(BASE_DIR, "logs", algo, exp_id, "training_log.jsonl")
    entry = _read_last_log_entry(log_file)
    if not entry:
        return "Episode 0/5000 | Reward: n/a | Avg100: n/a"
    ep = int(entry.get("episode", 0) or 0)
    reward = entry.get("reward_total", "n/a")
    avg100 = entry.get("reward_rolling_avg_100", "n/a")
    return f"Episode {ep}/{MAX_EPISODES} | Reward: {reward} | Avg100: {avg100}"

def all_experiments_done(max_episodes=MAX_EPISODES):
    for algo in ALGORITHMS:
        for exp_id in EXPERIMENT_TAGS:
            if not is_experiment_complete(algo, exp_id, max_episodes=max_episodes):
                return False
    return True

print("Total experiments:", len(ALGORITHMS) * len(EXPERIMENT_TAGS))

In [ ]:
BATCH_INDEX = 0  # Change manually: 0, 1, 2, 3
algo, start_idx, num_exp = BATCHES[BATCH_INDEX]
batch_number = BATCH_INDEX + 1

selected_experiments = EXPERIMENT_TAGS[start_idx:start_idx + num_exp]

print(f"Selected batch: {batch_number}/{len(BATCHES)}")
print(f"Algorithm: {algo}")
print(f"Experiment count: {len(selected_experiments)}")
for i, exp_id in enumerate(selected_experiments, start=1):
    status = "DONE" if is_experiment_complete(algo, exp_id, MAX_EPISODES) else "PENDING"
    print(f"[{algo.upper()}][Batch {batch_number}/{len(BATCHES)}] -> {exp_id} ({i}/{len(selected_experiments)}) [{status}]")

In [ ]:
batch_failures = []
batch_completed = 0
batch_skipped = 0

for i, exp_id in enumerate(selected_experiments, start=1):
    print("\n" + "-" * 80)
    print(f"[{algo.upper()}][Batch {batch_number}/{len(BATCHES)}] -> {exp_id} (Exp {i}/{len(selected_experiments)})")

    if is_experiment_complete(algo, exp_id, MAX_EPISODES):
        print(f"Skipping {exp_id} (already completed)")
        print(last_progress_line(algo, exp_id))
        batch_skipped += 1
        continue

    try:
        single_index = start_idx + (i - 1)
        cmd = f"""
python run_experiments.py \
--algo {algo} \
--max-episodes {MAX_EPISODES} \
--start-index {single_index} \
--max-experiments 1 \
--headless \
--resume
"""

        print("Command:")
        print(cmd)
        !{cmd}

        if not is_experiment_complete(algo, exp_id, MAX_EPISODES):
            raise RuntimeError("Completion criteria not met after run")

        print(last_progress_line(algo, exp_id))
        prune_experiment_models(algo, exp_id)
        batch_completed += 1

    except Exception as e:
        print(f"[ERROR] {exp_id} failed: {e}")
        batch_failures.append((exp_id, str(e)))
        continue

print("\n" + "=" * 80)
print(f"Batch summary: completed={batch_completed}, skipped={batch_skipped}, failed={len(batch_failures)}")
if batch_failures:
    for exp_id, err in batch_failures:
        print(f" - {exp_id}: {err}")

In [ ]:
if all_experiments_done():
    print("All 24 experiments completed. Running comparison plotting...")
    plot_cmd = f"""
python plot_metrics.py \
--compare-algos
"""
    !{plot_cmd}
    print("Plotting complete.")
else:
    print("Skipping plotting (incomplete runs)")

In [ ]:
Path(BASE_DIR, "results").mkdir(parents=True, exist_ok=True)
shutil.make_archive('/kaggle/working/final_results', 'zip', '/kaggle/working/results')
print('Saved: /kaggle/working/final_results.zip')